# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [11]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [12]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [13]:
#Cleaning Data 

spaceship.isna().sum()
spaceship = spaceship.dropna()

spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
spaceship

spaceship = spaceship.drop(columns=["PassengerId", "Name"])
spaceship

spaceship_dummies = pd.get_dummies(spaceship, dtype=int)
spaceship_dummies

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Earth,HomePlanet_Europa,HomePlanet_Mars,...,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_55 Cancri e,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_False,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,0,1,0,...,0,0,0,0,0,0,0,1,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,True,1,0,0,...,0,0,1,0,0,0,0,1,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,0,1,0,...,0,0,0,0,0,0,0,1,0,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,0,1,0,...,0,0,0,0,0,0,0,1,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,True,1,0,0,...,0,0,1,0,0,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,False,0,1,0,...,0,0,0,0,0,1,0,0,0,1
8689,18.0,0.0,0.0,0.0,0.0,0.0,False,1,0,0,...,0,0,0,1,0,0,1,0,1,0
8690,26.0,0.0,0.0,1872.0,1.0,0.0,True,1,0,0,...,0,0,0,1,0,0,0,1,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,False,0,1,0,...,0,1,0,0,0,1,0,0,1,0


**Perform Train Test Split**

In [17]:

# -------------------------
# TRAIN TEST SPLIT
# -------------------------

features = spaceship_dummies.drop(columns=["Transported"])
target = spaceship_dummies["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=0
)

# -------------------------
# FEATURE SCALING
# -------------------------

scaler = MinMaxScaler()

scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_test_norm = scaler.transform(X_test)

# Convert back to DataFrame
X_train_norm = pd.DataFrame(
    X_train_norm,
    columns=X_train.columns,
    index=X_train.index
)

X_test_norm = pd.DataFrame(
    X_test_norm,
    columns=X_test.columns,
    index=X_test.index
)

# -------------------------
# FEATURE SELECTION
# -------------------------

# Correlation matrix
corr_matrix = X_train_norm.corr().abs()

# Upper triangle matrix
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Find highly correlated features
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]

# Remove them
X_train_selected = X_train_norm.drop(columns=to_drop)
X_test_selected = X_test_norm.drop(columns=to_drop)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    bootstrap=True,   # True = Bagging, False = Pasting
    random_state=0
)

bag_model.fit(X_train_selected, y_train)

y_pred_bag = bag_model.predict(X_test_selected)

bag_acc = accuracy_score(y_test, y_pred_bag)

print("Bagging Accuracy:", bag_acc)

Bagging Accuracy: 0.7866868381240545


- Random Forests

In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=0
)

rf_model.fit(X_train_selected, y_train)

y_pred_rf = rf_model.predict(X_test_selected)

rf_acc = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", rf_acc)

Random Forest Accuracy: 0.7957639939485628


- Gradient Boosting

In [20]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=0
)

gb_model.fit(X_train_selected, y_train)

y_pred_gb = gb_model.predict(X_test_selected)

gb_acc = accuracy_score(y_test, y_pred_gb)

print("Gradient Boosting Accuracy:", gb_acc)

Gradient Boosting Accuracy: 0.7859304084720121


- Adaptive Boosting

In [21]:
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=0
)

ada_model.fit(X_train_selected, y_train)

y_pred_ada = ada_model.predict(X_test_selected)

ada_acc = accuracy_score(y_test, y_pred_ada)

print("AdaBoost Accuracy:", ada_acc)

AdaBoost Accuracy: 0.745839636913767


Which model is the best and why?

In [22]:
print("Model Accuracies:")
print("Bagging:", bag_acc)
print("Random Forest:", rf_acc)
print("Gradient Boosting:", gb_acc)
print("AdaBoost:", ada_acc)

Model Accuracies:
Bagging: 0.7866868381240545
Random Forest: 0.7957639939485628
Gradient Boosting: 0.7859304084720121
AdaBoost: 0.745839636913767


Best Model is Random Forest with an accuracy of 79.6%. Random Forest performed the best because it combines multiple decision trees trained on random subsets of the data and features. This reduces variance and overfitting while improving generalization compared to a single model.